In [ ]:
# Keep notebook paths stable after moving notebooks into notebooks/.
from pathlib import Path
import os
import sys

repo_root = Path.cwd()
if not (repo_root / "python_files").exists() and (repo_root.parent / "python_files").exists():
    repo_root = repo_root.parent
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


# Generate a LaTeX Diff Report (v2)

Fresh regenerated notebook. Use this v2 file if Jupyter kept running stale cells from the older Step 8 notebook.


## Rules

- `main_tex` must be the same relative path inside the old and new projects or zips.
- Use `bib = None` unless both old and new project zips include the needed `.bib` files.
- Use `bib = "bibtex"` for ordinary `.bib`/natbib workflows when the `.bib` files are present in both projects.
- Use `bib = "biber"` for biber workflows when the `.bib` files are present in both projects.
- Local mode assumes MiKTeX provides `latexdiff`, `pdflatex`, and `bibtex`/`biber`. MiKTeX's `latexdiff` also needs Perl.
- `latexdiff_engine = "online"` sends only the old/new main `.tex` text to https://3142.nl/latex-diff/ and then compiles the returned diff locally.
- Docker Desktop is needed only when `latexdiff_engine = "docker"`.

In [1]:
# ---------- User inputs ----------

# Set True for a tiny self-test project created by this notebook.
# Keep False for your real old/new manuscript packages.
use_demo_projects = False

# Your current repo layout uses old/old.zip and new/new.zip.
# You can also point these directly to zip files or expanded LaTeX folders.
old_project = r"old"
new_project = r"new"

# This file must exist inside both old and new project zips/folders.
main_tex = "manuscript.tex"

# Use None, "bibtex", or "biber".
# Keep None unless both old and new project zips include the required .bib files.
bib = None

# None uses blue additions and red struck-through deletions.
# You can also use a latexdiff style string such as "UNDERLINE", "CFONT", or "BOLD".
style = None

# Extra options forwarded to git-latexdiff/latexdiff, for example "--math-markup=whole".
other_cmdlines = ""

# Choose "online", "local", or "docker".
# online sends old/new main .tex text to https://3142.nl/latex-diff/ and compiles the returned diff locally.
# local uses MiKTeX/latexdiff from PATH.
# docker uses the upstream git-latexdiff-web worker image: am009/latexdiff-web-worker.
latexdiff_engine = "online"
confirm_online_upload = True
online_latexdiff_url = "https://3142.nl/latex-diff/"
online_latexdiff_timeout_seconds = 120

valid_latexdiff_engines = {"online", "local", "docker"}
if latexdiff_engine not in valid_latexdiff_engines:
    raise ValueError(f"latexdiff_engine must be one of {sorted(valid_latexdiff_engines)}")
use_docker = latexdiff_engine == "docker"
use_online_latexdiff = latexdiff_engine == "online"

run_docker_through_cmd = True
auto_start_docker = True
docker_start_timeout_seconds = 120
docker_desktop_exe = r"C:\Program Files\Docker\Docker\Docker Desktop.exe"

# Keep True to generate diff.pdf when you run all cells.
# Set False only when you want to validate paths/config without running latexdiff.
run_worker = True
# Set True to run `docker pull am009/latexdiff-web-worker` before the worker.
pull_image = False
debug = False
show_build_log = False

# Local tool names. Change these only if they are not on PATH.
latexdiff_executable = "latexdiff"
pdflatex_executable = "pdflatex"
bibtex_executable = "bibtex"
biber_executable = "biber"

# A fresh subfolder is created inside this folder on every run.
workspace_root = r"latexdiff_runs/notebook_runs"

In [2]:
import json
import shutil
import subprocess
import time
import zipfile
from datetime import datetime
from pathlib import Path

from python_files.latexdiff_web import (
    format_command,
    prepare_latexdiff_workspace,
    run_local_latexdiff,
    run_latexdiff_worker,
    run_online_latexdiff,
)

In [3]:
def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def reset_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def create_demo_projects(base_dir="latexdiff_runs/demo_projects"):
    base = reset_dir(base_dir)
    old_dir = base / "old_project"
    new_dir = base / "new_project"

    old_tex = r"""\documentclass{article}
\usepackage[utf8]{inputenc}
\begin{document}
\section{Introduction}
This is the original manuscript sentence.
The method uses a simple baseline model.
\section{Results}
The initial result is acceptable.
\end{document}
"""

    new_tex = r"""\documentclass{article}
\usepackage[utf8]{inputenc}
\begin{document}
\section{Introduction}
This is the revised manuscript sentence.
The method uses a simple baseline model and a stronger comparison.
\section{Results}
The updated result is substantially improved.
\section{Conclusion}
This short example checks the Pub Assist latexdiff workflow.
\end{document}
"""

    write_text(old_dir / "main.tex", old_tex)
    write_text(new_dir / "main.tex", new_tex)
    return old_dir, new_dir, "main.tex"


def normalize_zip_names(zip_path):
    with zipfile.ZipFile(zip_path) as archive:
        return sorted(name.replace("\\", "/") for name in archive.namelist() if not name.endswith("/"))


def zip_contains_main(zip_path, main_tex):
    expected = main_tex.replace("\\", "/")
    return expected in set(normalize_zip_names(zip_path))


def folder_contains_main(folder, main_tex):
    return (Path(folder) / main_tex).exists()


def resolve_project_input(project_path, label, main_tex):
    """Resolve a project folder, a zip file, or a folder containing one usable zip."""
    path = Path(project_path)

    if path.is_file() and path.suffix.lower() == ".zip":
        if not zip_contains_main(path, main_tex):
            raise FileNotFoundError(f"{label}: {main_tex!r} was not found inside {path}")
        return path

    if path.is_dir():
        if folder_contains_main(path, main_tex):
            return path

        candidate_zips = sorted(path.glob("*.zip"))
        usable_zips = [zip_path for zip_path in candidate_zips if zip_contains_main(zip_path, main_tex)]

        if len(usable_zips) == 1:
            print(f"{label}: using zip found inside folder: {usable_zips[0]}")
            return usable_zips[0]

        if len(usable_zips) > 1:
            raise ValueError(
                f"{label}: multiple zips in {path} contain {main_tex!r}. "
                "Point old_project/new_project directly to the intended zip."
            )

        available = ", ".join(str(zip_path) for zip_path in candidate_zips) or "no zip files"
        raise FileNotFoundError(
            f"{label}: could not find {main_tex!r} directly under {path}, "
            f"and no zip in that folder contains it. Available zips: {available}"
        )

    raise FileNotFoundError(f"{label}: input path does not exist: {path}")


if use_demo_projects:
    old_project, new_project, main_tex = create_demo_projects()
    bib = None
    print("Demo projects created.")

resolved_old_project = resolve_project_input(old_project, "old_project", main_tex)
resolved_new_project = resolve_project_input(new_project, "new_project", main_tex)

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
workspace_dir = Path(workspace_root) / run_id

print("Resolved inputs:")
print(f"old_project -> {resolved_old_project}")
print(f"new_project -> {resolved_new_project}")
print(f"main_tex -> {main_tex}")
print(f"workspace_dir -> {workspace_dir}")

old_project: using zip found inside folder: old\old.zip
new_project: using zip found inside folder: new\new.zip
Resolved inputs:
old_project -> old\old.zip
new_project -> new\new.zip
main_tex -> manuscript.tex
workspace_dir -> latexdiff_runs\notebook_runs\20260602_094741


In [4]:
workspace = prepare_latexdiff_workspace(
    old_project=resolved_old_project,
    new_project=resolved_new_project,
    main_tex=main_tex,
    workspace_dir=workspace_dir,
    bib=bib,
    style=style,
    other_cmdlines=other_cmdlines,
    overwrite=True,
)

workspace_path = Path(workspace["workspace"])

print("Prepared git-latexdiff-web workspace:")
for key in ["workspace", "old_zip", "new_zip", "config_json"]:
    print(f"- {key}: {workspace[key]}")

print("\nconfig.json:")
print(json.dumps(workspace["config"], indent=2))

print("\nSelected engine:", latexdiff_engine)
if latexdiff_engine == "docker":
    print("Docker command:")
    if run_docker_through_cmd and shutil.which("cmd"):
        print(format_command(["cmd", "/c"] + workspace["docker_command"]))
    else:
        print(format_command(workspace["docker_command"]))
elif latexdiff_engine == "online":
    print(f"Online service: {online_latexdiff_url}")
    print("Only old/new main .tex contents are uploaded; project zips, figures, .bib, and .bbl are not uploaded.")
else:
    print("Local mode will run latexdiff and pdflatex from PATH.")

Prepared git-latexdiff-web workspace:
- workspace: E:\github_codes\pub_assist\pub_assist\latexdiff_runs\notebook_runs\20260602_094741
- old_zip: E:\github_codes\pub_assist\pub_assist\latexdiff_runs\notebook_runs\20260602_094741\old.zip
- new_zip: E:\github_codes\pub_assist\pub_assist\latexdiff_runs\notebook_runs\20260602_094741\new.zip
- config_json: E:\github_codes\pub_assist\pub_assist\latexdiff_runs\notebook_runs\20260602_094741\config.json

config.json:
{
  "other_cmdlines": "",
  "style": {
    "new_text": {
      "color": [
        0,
        0,
        255
      ],
      "style": "underline_wave"
    },
    "old_text": {
      "color": [
        255,
        0,
        0
      ],
      "style": "strikeout"
    }
  },
  "main_tex": "manuscript.tex",
  "bib": "bibtex"
}

Selected engine: Docker
Docker command:
cmd /c docker run --rm -v E:\github_codes\pub_assist\pub_assist\latexdiff_runs\notebook_runs\20260602_094741:/work am009/latexdiff-web-worker


In [5]:
old_files = normalize_zip_names(workspace_path / "old.zip")
new_files = normalize_zip_names(workspace_path / "new.zip")

expected_main = main_tex.replace("\\", "/")
assert expected_main in old_files, f"old.zip does not contain {expected_main}"
assert expected_main in new_files, f"new.zip does not contain {expected_main}"

print(f"Validated old.zip: found {expected_main} and {len(old_files)} file(s).")
print(f"Validated new.zip: found {expected_main} and {len(new_files)} file(s).")

old_bibs = [name for name in old_files if name.lower().endswith((".bib", ".bbl"))]
new_bibs = [name for name in new_files if name.lower().endswith((".bib", ".bbl"))]
print(f"old bibliography files: {old_bibs if old_bibs else 'none'}")
print(f"new bibliography files: {new_bibs if new_bibs else 'none'}")

Validated old.zip: found manuscript.tex and 14 file(s).
Validated new.zip: found manuscript.tex and 12 file(s).
old bibliography files: ['references.bib']
new bibliography files: ['references_clean.bib']


In [6]:
def run_preflight_command(command, use_cmd=False):
    actual_command = ["cmd", "/c"] + command if use_cmd else command
    return subprocess.run(
        actual_command,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
    )



def docker_info_result(use_cmd=False):
    return run_preflight_command(["docker", "info"], use_cmd=use_cmd)


def try_start_docker_desktop(timeout_seconds=120):
    docker_desktop_path = Path(docker_desktop_exe)

    if not docker_desktop_path.exists():
        print(f"Docker Desktop executable not found at: {docker_desktop_path}")
        return False

    print(f"Starting Docker Desktop: {docker_desktop_path}")
    creationflags = getattr(subprocess, "CREATE_NO_WINDOW", 0)
    subprocess.Popen(
        [str(docker_desktop_path)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        creationflags=creationflags,
    )

    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        result = docker_info_result(use_cmd=docker_use_cmd)
        if result.returncode == 0:
            print("Docker Desktop is ready.")
            return True
        print("Waiting for Docker Desktop to become ready...")
        time.sleep(5)

    return False


docker_use_cmd = False

if use_online_latexdiff:
    if not confirm_online_upload:
        raise RuntimeError("Online latexdiff is selected, but confirm_online_upload=False. Set it True to upload old/new main .tex text.")
    print(f"Online latexdiff enabled: {online_latexdiff_url}")
    print("Only the old and new main .tex files will be sent to the online service.")

if use_docker:
    if shutil.which("docker") is None:
        raise RuntimeError("Docker CLI was not found on PATH. Install Docker Desktop or set latexdiff_engine='online' or 'local'.")

    docker_use_cmd = bool(run_docker_through_cmd and shutil.which("cmd"))
    if run_docker_through_cmd and not docker_use_cmd:
        print("cmd.exe was not found, so Docker will be run directly through Python subprocess.")

    version_result = run_preflight_command(["docker", "--version"], use_cmd=docker_use_cmd)
    if version_result.returncode != 0:
        raise RuntimeError(
            "Docker CLI exists but `docker --version` failed.\n\n"
            f"stdout:\n{version_result.stdout}\n\nstderr:\n{version_result.stderr}"
        )
    print(version_result.stdout.strip())

    info_result = docker_info_result(use_cmd=docker_use_cmd)
    if info_result.returncode != 0 and auto_start_docker:
        started = try_start_docker_desktop(timeout_seconds=docker_start_timeout_seconds)
        if started:
            info_result = docker_info_result(use_cmd=docker_use_cmd)

    if info_result.returncode != 0:
        raise RuntimeError(
            "Docker CLI is installed, but the Docker daemon is not available. "
            "The notebook tried to start Docker Desktop first if auto_start_docker=True. "
            "Start Docker Desktop manually or increase docker_start_timeout_seconds, then rerun.\n\n"
            f"stdout:\n{info_result.stdout}\n\nstderr:\n{info_result.stderr}"
        )

    print("Docker daemon is available.")
    print("Docker execution mode:", "cmd.exe /c docker" if docker_use_cmd else "direct docker subprocess")
else:
    print("Docker preflight skipped because latexdiff_engine is not 'docker'.")

Docker version 29.2.1, build a5c7197
Starting Docker Desktop: C:\Program Files\Docker\Docker\Docker Desktop.exe
Waiting for Docker Desktop to become ready...
Docker Desktop is ready.
Docker daemon is available.
Docker execution mode: cmd.exe /c docker


In [7]:
if run_worker:
    if latexdiff_engine == "docker":
        result = run_latexdiff_worker(
            workspace_dir=workspace_path,
            debug=debug,
            pull_image=pull_image,
            use_cmd=docker_use_cmd,
            check=False,
        )
        engine_name = "Docker worker"
    elif latexdiff_engine == "online":
        result = run_online_latexdiff(
            workspace_dir=workspace_path,
            service_url=online_latexdiff_url,
            pdflatex_executable=pdflatex_executable,
            timeout=online_latexdiff_timeout_seconds,
            check=False,
        )
        engine_name = "online latexdiff"
    else:
        result = run_local_latexdiff(
            workspace_dir=workspace_path,
            latexdiff_executable=latexdiff_executable,
            pdflatex_executable=pdflatex_executable,
            bibtex_executable=bibtex_executable,
            biber_executable=biber_executable,
            check=False,
        )
        engine_name = "local MiKTeX/latexdiff"

    print(f"{engine_name} return code: {result['returncode']}")
    if result.get("docker_image"):
        print(f"Docker image: {result['docker_image']}")
    if result.get("docker_pull_command"):
        print(f"Docker pull command: {result['docker_pull_command']}")
    if result.get("artifact_files"):
        print("\nArtifacts found:")
        for artifact in result["artifact_files"]:
            print(f"- {artifact}")

    should_show_log = show_build_log or result["returncode"] != 0

    if should_show_log and result["stdout"]:
        print("\nstdout:\n")
        print(result["stdout"])
    if should_show_log and result["stderr"]:
        print("\nstderr:\n")
        print(result["stderr"])

    diff_pdf = Path(result["diff_pdf"])
    diff_tex = Path(result.get("diff_tex", workspace_path / "git-latexdiff" / "new" / main_tex))

    if result["returncode"] != 0:
        raise RuntimeError("Latexdiff failed. Read stdout/stderr and artifact paths above for the LaTeX error.")
    if not diff_pdf.exists():
        raise RuntimeError("Latexdiff finished without creating diff.pdf. Read stdout/stderr and artifact paths above for the LaTeX error.")

    print("\nLatexdiff completed successfully.")
    print(f"diff.pdf: {diff_pdf}")
    print(f"diffed main tex: {diff_tex}")
    if not show_build_log:
        print("Set show_build_log=True if you want to see the full LaTeX log.")
else:
    print("run_worker is False. Set run_worker=True after this validation passes.")

Docker worker return code: 0

Latexdiff completed successfully.
diff.pdf: E:\github_codes\pub_assist\pub_assist\latexdiff_runs\notebook_runs\20260602_094741\diff.pdf
diffed main tex: E:\github_codes\pub_assist\pub_assist\latexdiff_runs\notebook_runs\20260602_094741\git-latexdiff\new\manuscript.tex
Set show_build_log=True if you want to see the full LaTeX log.


In [8]:
diff_pdf_path = workspace_path / "diff.pdf"

if latexdiff_engine == "docker":
    diff_tex_path = workspace_path / "git-latexdiff" / "new" / main_tex
elif latexdiff_engine == "online":
    diff_tex_path = workspace_path / "online-latexdiff" / "build" / main_tex
else:
    diff_tex_path = workspace_path / "local-latexdiff" / "build" / main_tex

expected_outputs = [
    diff_pdf_path,
    diff_tex_path,
]

for path in expected_outputs:
    print(f"{path}: {'FOUND' if path.exists() else 'missing'}")

E:\github_codes\pub_assist\pub_assist\latexdiff_runs\notebook_runs\20260602_094741\diff.pdf: FOUND
E:\github_codes\pub_assist\pub_assist\latexdiff_runs\notebook_runs\20260602_094741\git-latexdiff\new\manuscript.tex: FOUND
